In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

possible_train_paths = [
    Path("data/jigsaw-unintended-bias-in-toxicity-classification/train.csv"),
    Path("/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/train.csv"),
    Path(
        "/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/train.csv.zip"
    ),
    Path("kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/train.csv"),
]
possible_test_paths = [
    Path("data/jigsaw-unintended-bias-in-toxicity-classification/test.csv"),
    Path("/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/test.csv"),
    Path(
        "/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/test.csv.zip"
    ),
    Path("kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/test.csv"),
]


def locate_file(possible_paths: list) -> Path:
    for p in possible_paths:
        if p.is_file():
            return p
    raise FileNotFoundError(f"None of the expected files were found: {possible_paths}")


TRAIN_PATH = locate_file(possible_train_paths)
TEST_PATH = locate_file(possible_test_paths)

train_df = pd.read_csv(TRAIN_PATH, usecols=["id", "target", "comment_text"])
test_df = pd.read_csv(TEST_PATH, usecols=["id", "comment_text"])

train_df["comment_text"] = train_df["comment_text"].fillna("")
test_df["comment_text"] = test_df["comment_text"].fillna("")



In [2]:
vectorizer = TfidfVectorizer(
    max_features=100_000,  # fewer dimensions → faster fit/transform
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
)

# Use fit_transform to avoid a second full pass over the training text.
X_full_vec = vectorizer.fit_transform(train_df["comment_text"]).astype(np.float32)
test_X = vectorizer.transform(test_df["comment_text"]).astype(np.float32)

full_y = (train_df["target"].values >= 0.5).astype(np.int8)

final_model = LogisticRegression(
    solver="saga",
    max_iter=100,  # fewer passes over data
    n_jobs=-1,
    class_weight="balanced",
    verbose=0,
)

final_model.fit(X_full_vec, full_y)

train_pred = final_model.predict_proba(X_full_vec)[:, 1]
train_auc = roc_auc_score(full_y, train_pred)
print(f"Training ROC‑AUC (full data): {train_auc:.5f}")



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training ROC‑AUC (full data): 0.96704


In [3]:
test_pred = final_model.predict_proba(test_X)[:, 1]

submission = pd.DataFrame({"id": test_df["id"], "prediction": test_pred})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file '{submission_path}' written with {len(submission)} rows.")

Submission file 'submission.csv' written with 97320 rows.
